# 04 — Bot vs. theory validation (live vs. replay)

**Goal:** measure rigorously whether the bot executes the theoretical model faithfully.

Two sources are compared over the SAME time window:
- **REAL**: the MT5 account history (what the bot actually did)
- **REPLAY**: the model run offline on the price CSV (what the model *says* should be done)

This notebook answers **a single question**: *is the bot a faithful executor of the model?*
It does NOT answer *"is the model profitable live?"* — that would need hundreds of trades and many weeks.

### The 4 checks
1. **Decision parity** — did the bot choose the same direction as the model? (deterministic)
2. **Outcome parity** — do TP/SL/TIMEOUT outcomes match?
3. **Slippage** — how much, and in which direction, does the entry price differ? Is it biased?
4. **Reward** — does the real per-trade return resemble the theoretical one?

### Preparing the inputs
- **REAL**: in MT5 → *History* tab → right click → *Report* → save as xlsx. It must contain a `Positions` sheet. The column names mapped in cell 2 (`Fecha/Hora`, `Posición`, `Precio`, …) are those of a **Spanish-language** MT5 terminal; adjust the mapping if your terminal uses another language.
- **REPLAY**: first create the model artifact with `python scripts/export_model.py`, then run `src/replay_to_xlsx.py` **over the same window** the bot traded (from `src/`: `python replay_to_xlsx.py --start-date YYYY-MM-DD`). It writes `outputs/replay.xlsx`, with the `replay` and `trades_only` sheets.

### Note on reproducibility
This notebook is committed **without outputs** because its first input — the demo account's trade-history export — is not included in the repository (it identifies the broker account). The live-validation figures quoted in the README, `results/summary.md` and the technical report were produced by running this notebook on that export.

In [ ]:
# === Cell 1: Configuration and imports ===
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 40)

# === PATHS — adjust if needed ===
REAL_PATH   = pathlib.Path('../data/USDJPY_real_account.xlsx')   # MT5 account export (not included)
REPLAY_PATH = pathlib.Path('../outputs/replay.xlsx')             # output of src/replay_to_xlsx.py
REAL_SHEET    = 'Positions'
REPLAY_SHEET  = 'trades_only'

# Symbol parameters
PIP = 0.01          # 1 USDJPY pip = 0.01
TICK = 0.001        # tick size

print('Paths configured.')
print(f'  REAL:   {REAL_PATH}')
print(f'  REPLAY: {REPLAY_PATH}')

In [ ]:
# === Cell 2: Load and normalize REAL (MT5 export) ===
real = pd.read_excel(REAL_PATH, sheet_name=REAL_SHEET)

# Map the MT5 column names (Spanish-language terminal export) to internal names
real = real.rename(columns={
    'Fecha/Hora':   'open_dt',
    'Posición':     'ticket',
    'Símbolo':      'symbol',
    'Tipo':         'type',
    'Volumen ':     'lot',      # note: MT5 sometimes leaves a trailing space
    'Volumen':      'lot',
    'Precio':       'open_price',
    'S / L':        'sl',
    'T / P':        'tp',
    'Fecha/Hora.1': 'close_dt',
    'Precio.1':     'close_price',
    'Comisión':     'commission',
    'Swap':         'swap',
    'Beneficio':    'pnl',
})

real['open_dt']  = pd.to_datetime(real['open_dt'])
real['close_dt'] = pd.to_datetime(real['close_dt'])
real['direction'] = real['type'].str.lower().map({'buy': 'LONG', 'sell': 'SHORT'})
for c in ['open_price','close_price','sl','tp','pnl','lot','commission','swap']:
    if c in real.columns:
        real[c] = pd.to_numeric(real[c], errors='coerce')

# Entry bar = 15M floor of the opening time (to match against the replay)
real['entry_bar'] = real['open_dt'].dt.floor('15min')

print(f'REAL: {len(real)} positions')
print(f'  Period: {real["open_dt"].min()} → {real["open_dt"].max()}')
print(f'  Directions: {real["direction"].value_counts().to_dict()}')
print(f'  Lots: {sorted(real["lot"].dropna().unique())}')
print(f'  Total PnL: ${real["pnl"].sum():+.2f}')
real.head()

In [ ]:
# === Cell 3: Load and normalize REPLAY ===
replay = pd.read_excel(REPLAY_PATH, sheet_name=REPLAY_SHEET)
replay['Datetime'] = pd.to_datetime(replay['Datetime'])
# In the replay, 'Datetime' is the ENTRY bar (entry at that bar's Open)
replay = replay.rename(columns={'Datetime': 'entry_bar', 'decision': 'direction',
                                 'Open': 'replay_open'})

print(f'REPLAY: {len(replay)} trades')
print(f'  Period: {replay["entry_bar"].min()} → {replay["entry_bar"].max()}')
print(f'  Directions: {replay["direction"].value_counts().to_dict()}')
print(f'  Outcomes: {replay["outcome"].value_counts().to_dict()}')
replay[['entry_bar','direction','prob','replay_open','outcome','exit_price','reward_pct']].head()

In [ ]:
# === Cell 4: Check the time overlap ===
# Comparing only makes sense over the period where BOTH sources have data.
overlap_start = max(real['entry_bar'].min(), replay['entry_bar'].min())
overlap_end   = min(real['entry_bar'].max(), replay['entry_bar'].max())
print(f'Overlap window: {overlap_start} → {overlap_end}')

real_ov   = real[(real['entry_bar'] >= overlap_start) & (real['entry_bar'] <= overlap_end)].copy()
replay_ov = replay[(replay['entry_bar'] >= overlap_start) & (replay['entry_bar'] <= overlap_end)].copy()
print(f'  REAL in window:   {len(real_ov)} trades')
print(f'  REPLAY in window: {len(replay_ov)} trades')

if len(real_ov) == 0 or len(replay_ov) == 0:
    print('\n⚠️  No overlap — check that the replay covers the live period (use --start-date).')

In [ ]:
# === Cell 5: MATCH by entry bar ===
# The bot enters during bar E; the replay marks that same bar E.
# Trades are matched exactly on entry_bar. Duplicate trades in the same bar are reported.

dup_real = real_ov['entry_bar'].duplicated().sum()
dup_replay = replay_ov['entry_bar'].duplicated().sum()
if dup_real or dup_replay:
    print(f'Note: bars with multiple trades — REAL: {dup_real}, REPLAY: {dup_replay}')
    print('  (The bot can stack several entries; the match keeps the first one on each side.)')

real_m = real_ov.drop_duplicates('entry_bar', keep='first').set_index('entry_bar')
replay_m = replay_ov.drop_duplicates('entry_bar', keep='first').set_index('entry_bar')

matched = real_m.join(replay_m, how='inner', lsuffix='_real', rsuffix='_replay')
only_real = real_m.index.difference(replay_m.index)
only_replay = replay_m.index.difference(real_m.index)

print(f'\n=== MATCH RESULT ===')
print(f'  Matched (both):          {len(matched)}')
print(f'  REAL only (no replay):   {len(only_real)}')
print(f'  REPLAY only (no real):   {len(only_replay)}')

cov = len(matched) / len(real_m) * 100 if len(real_m) else 0
print(f'\n  Coverage: {cov:.1f}% of the real trades have a counterpart in the replay')

In [ ]:
# === Cell 6: Diagnose the UNMATCHED trades ===
# Why a real trade does not appear in the replay (or vice versa).
# Benign cause: the bot used a different .pkl earlier, or the replay does not cover that bar.
# Worrying cause: the bot trades on bars where the model says HOLD.

replay_full = pd.read_excel(REPLAY_PATH, sheet_name='replay')
replay_full['entry_bar'] = pd.to_datetime(replay_full['Datetime'])
replay_full = replay_full.set_index('entry_bar')

if len(only_real) > 0:
    print(f'=== {len(only_real)} REAL trades without a match — what did the model say on those bars? ===\n')
    rows = []
    for ts in only_real:
        if ts in replay_full.index:
            rr = replay_full.loc[ts]
            rows.append({'entry_bar': ts, 'real_dir': real_m.loc[ts,'direction'],
                         'model_decision': rr['decision'], 'model_prob': round(rr['prob'],4),
                         'skip_reason': rr.get('skip_reason','')})
        else:
            rows.append({'entry_bar': ts, 'real_dir': real_m.loc[ts,'direction'],
                         'model_decision': 'NO_BAR_IN_REPLAY', 'model_prob': np.nan, 'skip_reason': ''})
    diag = pd.DataFrame(rows)
    print(diag.to_string(index=False))
    # Summary
    print('\nWhat the model said on bars where the bot DID trade:')
    print(diag['model_decision'].value_counts().to_dict())
    n_hold = (diag['model_decision']=='HOLD').sum()
    if n_hold > 0:
        print(f'\n⚠️  {n_hold} real trades on bars where the model said HOLD.')
        print('    If there are MANY and they are recent → the bot is not using the correct .pkl.')
        print('    If they are from the earlier period → they came from a previous model (expected).')
else:
    print('Every real trade has a match. ✓')

---
## Check 1 — Decision parity

Did the bot choose the same direction as the model on the matched bars?
This is deterministic: if the bot and the replay use the same `.pkl` and the same features,
the direction must **always** agree. Any discrepancy signals a bug.

In [ ]:
# === Cell 7: Direction parity ===
matched['dir_match'] = matched['direction_real'] == matched['direction_replay']
n_match = matched['dir_match'].sum()
print(f'Direction parity: {n_match} / {len(matched)} = {n_match/len(matched)*100:.1f}%')

if n_match < len(matched):
    print('\n⚠️  Direction discrepancies (potential bug):')
    bad = matched[~matched['dir_match']][['direction_real','direction_replay','prob']]
    print(bad.to_string())
else:
    print('✓ 100% direction parity — the bot trades the correct side.')

---
## Check 2 — Outcome parity (TP / SL / TIMEOUT)

The real outcome is classified by comparing the closing price with the TP/SL levels,
and cross-tabulated against the outcome the replay predicted.

In [ ]:
# === Cell 8: Classify the real outcome and compare ===
def classify_real_outcome(r, tol=0.01):
    """Classifies whether the real close was TP, SL or OTHER (timeout/manual)."""
    if pd.isna(r['close_price']) or pd.isna(r['tp_real']) or pd.isna(r['sl_real']):
        return 'UNKNOWN'
    cp, tp, sl, d = r['close_price'], r['tp_real'], r['sl_real'], r['direction_real']
    if d == 'LONG':
        if cp >= tp - tol: return 'TP'
        if cp <= sl + tol: return 'SL'
    else:
        if cp <= tp + tol: return 'TP'
        if cp >= sl - tol: return 'SL'
    return 'OTHER'  # timeout or manual close

# Rename for clarity
matched['tp_real'] = matched['tp']
matched['sl_real'] = matched['sl']
matched['real_outcome'] = matched.apply(classify_real_outcome, axis=1)

# The replay uses TIMEOUT/TP/SL/TIE_AS_SL — TIE_AS_SL is normalized to SL for the comparison
matched['replay_outcome_norm'] = matched['outcome'].replace({'TIE_AS_SL':'SL'})
# OTHER in the real history ≈ TIMEOUT in the replay (time-based exit)
matched['real_outcome_norm'] = matched['real_outcome'].replace({'OTHER':'TIMEOUT'})

print('Outcome confusion matrix (real vs replay):')
ct = pd.crosstab(matched['real_outcome_norm'], matched['replay_outcome_norm'],
                 rownames=['REAL'], colnames=['REPLAY'])
print(ct)

agree = (matched['real_outcome_norm'] == matched['replay_outcome_norm']).sum()
print(f'\nOutcome parity: {agree} / {len(matched)} = {agree/len(matched)*100:.1f}%')
print('\nNote: TP↔TIMEOUT differences usually come from manual closes or from the bot being offline.')

---
## Check 3 — Slippage (the most important one for costs)

Slippage is measured **with its sign**, using the convention *positive = worse for the trader*.
- LONG: buying above the theoretical Open → `real_open − replay_open` > 0 is bad
- SHORT: selling below it → `replay_open − real_open` > 0 is bad

We then test whether the bias is significant and split it by direction,
because pure spread affects LONG and SHORT symmetrically.

In [ ]:
# === Cell 9: Signed slippage + significance ===
matched['slip_pips'] = np.where(
    matched['direction_real'] == 'LONG',
    (matched['open_price'] - matched['replay_open']) / PIP,
    (matched['replay_open'] - matched['open_price']) / PIP
)

slip = matched['slip_pips'].dropna().values
print('=== SLIPPAGE (positive = against the trader) ===')
print(f'  N:       {len(slip)}')
print(f'  Mean:    {slip.mean():+.3f} pips')
print(f'  Median:  {np.median(slip):+.3f} pips')
print(f'  Std:     {slip.std():.3f} pips')
print(f'  Min/Max: {slip.min():+.3f} / {slip.max():+.3f}')
print(f'  Favorable (<0): {(slip<0).sum()} | Neutral (=0): {(slip==0).sum()} | Adverse (>0): {(slip>0).sum()}')

# t-test: does the mean differ from zero?
if len(slip) >= 5:
    t, p = stats.ttest_1samp(slip, 0)
    print(f'\n  t-test mean=0:   t={t:.2f}, p={p:.2e}')
    # Bootstrap CI
    rng = np.random.default_rng(42)
    boot = np.array([slip[rng.integers(0,len(slip),len(slip))].mean() for _ in range(10000)])
    lo, hi = np.percentile(boot,[2.5,97.5])
    print(f'  95% CI of the mean: [{lo:+.3f}, {hi:+.3f}] pips')
    if lo > 0:
        print('  → SIGNIFICANT bias against the trader (expected: it is the spread).')
    elif hi < 0:
        print('  → Bias in the trader\'s favor (unusual — investigate).')
    else:
        print('  → Not distinguishable from zero.')

In [ ]:
# === Cell 10: Slippage by direction (pure spread or adverse selection?) ===
# With pure SPREAD, LONG and SHORT show a similar bias against the trader.
# If SHORT goes the opposite way from LONG, there is a sign or fill problem.
print('Slippage by direction:')
for d in ['LONG','SHORT']:
    sub = matched[matched['direction_real']==d]['slip_pips'].dropna()
    if len(sub) > 0:
        print(f'  {d}: n={len(sub):>3} | mean={sub.mean():+.3f} pips | median={sub.median():+.3f}')
    else:
        print(f'  {d}: no trades in the window')

print('''
How to read it:
  - USDJPY spread ~1-1.5 pips → half-spread ~0.5-0.75 pips per side.
  - If both directions show +0.5 to +0.75 → it is the SPREAD, not manipulation. Normal.
  - If SHORT has the opposite sign from LONG → review the price convention (bid/ask).
''')

---
## Check 4 — Reward per trade (real vs theoretical)

We compare the return per trade. The real return already includes the actual spread/slippage;
the replay already includes the doubled cost we model. They should be close.

In [ ]:
# === Cell 11: Real vs replay reward ===
# Real reward in price-percent (same as the model): (close-open)/open for a LONG
matched['real_reward_pct'] = np.where(
    matched['direction_real']=='LONG',
    (matched['close_price'] - matched['open_price'])/matched['open_price']*100,
    (matched['open_price'] - matched['close_price'])/matched['open_price']*100
)
# the replay's reward_pct is already in %

valid = matched.dropna(subset=['real_reward_pct','reward_pct'])
print('=== REWARD PER TRADE (%) ===')
print(f'  N: {len(valid)}')
print(f'  Real   — mean: {valid["real_reward_pct"].mean():+.4f}% | sum: {valid["real_reward_pct"].sum():+.2f}%')
print(f'  Replay — mean: {valid["reward_pct"].mean():+.4f}% | sum: {valid["reward_pct"].sum():+.2f}%')
print(f'  Difference in means: {(valid["real_reward_pct"].mean()-valid["reward_pct"].mean()):+.4f}% per trade')

# Trade-by-trade correlation
if len(valid) >= 5:
    r = np.corrcoef(valid['real_reward_pct'], valid['reward_pct'])[0,1]
    print(f'  Real vs replay correlation (trade by trade): {r:.3f}')
    print('    (high correlation = the same trade yields a similar reward; low = divergence)')

# Paired difference test
if len(valid) >= 5:
    t, p = stats.ttest_rel(valid['real_reward_pct'], valid['reward_pct'])
    print(f'  Paired t-test (real vs replay): t={t:.2f}, p={p:.3f}')
    print('    p>0.05 → no detectable systematic difference (good).')

In [ ]:
# === Cell 12: PnL in money — matched vs unmatched ===
# Do the trades that DO follow the model win, and the ones that DON'T lose?
matched_tickets = set(matched['ticket']) if 'ticket' in matched.columns else set()
real_ov2 = real_ov.copy()
real_ov2['is_matched'] = real_ov2['entry_bar'].isin(matched.index)

print('PnL in money:')
g = real_ov2.groupby('is_matched')['pnl'].agg(['count','sum','mean'])
g.index = ['Not matched','Matched']
print(g.round(2))
print('''
How to read it: if the matched trades win and the unmatched ones lose, that confirms
that following the model is correct, and that the off-model trades (from an old .pkl
or from HOLD bars) are the ones dragging performance down.
''')

---
## Charts

In [ ]:
# === Cell 13: Visualizations ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (1) Slippage histogram
ax = axes[0,0]
ax.hist(slip, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(0, color='green', ls='--', label='zero (ideal)')
ax.axvline(slip.mean(), color='red', ls='-', label=f'mean {slip.mean():+.2f}')
ax.set_title('Slippage (pips, + = against the trader)'); ax.set_xlabel('pips'); ax.legend()

# (2) Real vs replay reward scatter
ax = axes[0,1]
if len(valid) >= 2:
    ax.scatter(valid['reward_pct'], valid['real_reward_pct'], alpha=0.6)
    lim = [min(valid['reward_pct'].min(), valid['real_reward_pct'].min()),
           max(valid['reward_pct'].max(), valid['real_reward_pct'].max())]
    ax.plot(lim, lim, 'g--', label='perfect parity')
    ax.set_xlabel('replay reward %'); ax.set_ylabel('real reward %')
    ax.set_title('Reward per trade: real vs replay'); ax.legend()

# (3) Cumulative equity curve (real vs replay, in %)
ax = axes[1,0]
vs = valid.sort_index()
ax.plot(range(len(vs)), vs['real_reward_pct'].cumsum().values, label='REAL', marker='o', ms=3)
ax.plot(range(len(vs)), vs['reward_pct'].cumsum().values, label='REPLAY', marker='s', ms=3)
ax.set_title('Cumulative reward (%)'); ax.set_xlabel('trade #'); ax.set_ylabel('cum. %'); ax.legend()

# (4) Outcome parity bars
ax = axes[1,1]
oc = pd.DataFrame({
    'REAL': matched['real_outcome_norm'].value_counts(),
    'REPLAY': matched['replay_outcome_norm'].value_counts()
}).fillna(0)
oc.plot(kind='bar', ax=ax)
ax.set_title('Outcome counts'); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../outputs/validation_charts.png', dpi=110, bbox_inches='tight')
plt.show()
print('Charts saved to ../outputs/validation_charts.png')

---
## Final verdict

In [ ]:
# === Cell 14: Automatic verdict ===
print('='*70)
print('  VERDICT — IS THE BOT A FAITHFUL EXECUTOR OF THE MODEL?')
print('='*70)

checks = []

# 1. Direction
dir_ok = (matched['dir_match'].mean() >= 0.99)
checks.append(('Direction parity ≥99%', dir_ok,
               f'{matched["dir_match"].mean()*100:.1f}%'))

# 2. Outcome
out_rate = (matched['real_outcome_norm']==matched['replay_outcome_norm']).mean()
out_ok = out_rate >= 0.85
checks.append(('Outcome parity ≥85%', out_ok, f'{out_rate*100:.1f}%'))

# 3. Slippage explained by the spread (mean < 1.0 pip)
slip_ok = abs(slip.mean()) < 1.0
checks.append(('Mean slippage < 1.0 pip (≈spread)', slip_ok, f'{slip.mean():+.3f} pips'))

# 4. Reward with no systematic difference
if len(valid) >= 5:
    _, p_rew = stats.ttest_rel(valid['real_reward_pct'], valid['reward_pct'])
    rew_ok = p_rew > 0.05
    checks.append(('No systematic reward difference (p>0.05)', rew_ok, f'p={p_rew:.3f}'))

# 5. High reward correlation
if len(valid) >= 5:
    rcorr = np.corrcoef(valid['real_reward_pct'], valid['reward_pct'])[0,1]
    corr_ok = rcorr > 0.7
    checks.append(('Real-replay reward correlation >0.7', corr_ok, f'{rcorr:.3f}'))

print()
n_pass = 0
for name, ok, val in checks:
    mark = '✓' if ok else '✗'
    print(f'  [{mark}] {name:<45} {val}')
    n_pass += ok

print(f'\n  {n_pass}/{len(checks)} criteria met')
if n_pass == len(checks):
    print('\n  ✓ THE BOT EXECUTES THE MODEL FAITHFULLY.')
    print('    The differences are attributable to normal spread/slippage.')
elif n_pass >= len(checks)-1:
    print('\n  ~ HIGH FIDELITY with one observation. Review the failed criterion.')
else:
    print('\n  ✗ THERE ARE DIVERGENCES TO INVESTIGATE before trusting the bot live.')

print('''
Reminder: this notebook validates FIDELITY (bot = model), not PROFITABILITY.
Confirming a positive live edge requires hundreds of trades and many weeks.
''')